In [51]:
!pip install -q transformers==4.44.2 sentencepiece accelerate

### Import Libraries

In [52]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import pipeline

### Load dataset

In [53]:
df = pd.read_csv("/content/amazon_reviews_clustered.csv")

In [54]:
df.head()

,id,name,brand,categories,asins,reviews.rating,reviews.title,reviews.text,reviews.username,reviews.date,review_length,cluster_text,Cluster,Meta_Category
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Kindle,This product so far has not disappointed. My c...,Adapter,2017-01-13 00:00:00+00:00,27,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,very fast,great for beginner or experienced person. Boug...,truman,2017-01-13 00:00:00+00:00,14,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,DaveZ,2017-01-13 00:00:00+00:00,26,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,4.0,Good!!!,I've had my Fire HD 8 two weeks now and I love...,Shacks,2017-01-13 00:00:00+00:00,117,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Fantastic Tablet for kids,I bought this for my grand daughter when she c...,explore42,2017-01-12 00:00:00+00:00,117,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders


### Find the Top 3 Products in Each Category

In [55]:
product_stats = (
    df.groupby(["Meta_Category", "name"])
      .agg(
          avg_rating=("reviews.rating", "mean"),
          review_count=("reviews.rating", "count")
      )
      .reset_index()
)

### Keep products with enough reviews

In [56]:
product_stats = product_stats[
    product_stats["review_count"] >= 20
]

### Select Top 3 Products

In [57]:
top_products = (
    product_stats
    .sort_values(
        ["Meta_Category", "avg_rating"],
        ascending=[True, False]
    )
    .groupby("Meta_Category")
    .head(3)
)

top_products

,Meta_Category,name,avg_rating,review_count
0,Batteries & Household Essentials,AmazonBasics AA Performance Alkaline Batteries...,4.414875,3442
1,Batteries & Household Essentials,AmazonBasics AAA Performance Alkaline Batterie...,4.397141,7554
5,Electronics & Streaming Devices,Unknown Product,4.707278,5056
16,Smart Home & Alexa Devices,"Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",4.773438,128
8,Smart Home & Alexa Devices,Amazon - Echo Plus w/ Built-In Hub - Silver,4.748727,589
7,Smart Home & Alexa Devices,Amazon - Amazon Tap Portable Bluetooth and Wi-...,4.729560,318
68,Tablets & eReaders,Amazon Kindle Paperwhite - eBook reader - 4 GB...,4.755038,3176
123,Tablets & eReaders,"Kindle Voyage E-reader, 6 High-Resolution Disp...",4.743103,580
61,Tablets & eReaders,Amazon Fire Hd 8 8in Tablet 16gb Black B018szt...,4.740741,135


### Collect Reviews of Those Products

In [58]:
selected_reviews = df.merge(
    top_products[
        ["Meta_Category","name"]
    ],
    on=["Meta_Category","name"]
)

In [59]:
print("Total reviews:", len(selected_reviews))

Total reviews: 20978


### Split Reviews into Sentences

In [60]:
def split_sentences(text):

    text = str(text)

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if 15 < len(s.strip()) < 300
    ]

### TF-IDF Sentence Selection

In [61]:
def top_representative_sentences(texts, n=40):

    all_sentences = []

    for text in texts:

        all_sentences.extend(
            split_sentences(text)
        )

    if len(all_sentences)==0:
        return []

    # Remove duplicate sentences

    all_sentences = list(
        dict.fromkeys(all_sentences)
    )

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=2000
    )

    X = vectorizer.fit_transform(all_sentences)

    scores = np.asarray(
        X.sum(axis=1)
    ).ravel()

    top_idx = scores.argsort()[::-1][:n]

    return [
        all_sentences[i]
        for i in sorted(top_idx)
    ]

In [62]:
print(selected_reviews.head())

                     id                                               name  \
0  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
1  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
2  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
3  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
4  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   

    brand                                         categories       asins  \
0  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
1  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
2  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
3  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
4  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   

   reviews.rating          reviews.title  \
0             3.0      I like 

In [63]:
category = "Tablets & eReaders"

texts = selected_reviews.loc[
    selected_reviews["Meta_Category"] == category,
    "reviews.text"
]

### Create the input text

In [64]:
selected_sentences = top_representative_sentences(texts, n=40)

### Split into chunks

In [65]:
chunks = chunk_sentences(selected_sentences, chunk_size=10)

print("Number of chunks:", len(chunks))

Number of chunks: 4


### Load T5

In [66]:
summarizer = pipeline(
    "summarization",
    model="t5-small",
    device=-1
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Summarize each chunk

In [67]:
chunk_summaries = []

for chunk in chunks:

    text = "summarize: " + " ".join(chunk)

    summary = summarizer(
        text,
        max_length=120,
        min_length=40,
        do_sample=False
    )[0]["summary_text"]

    chunk_summaries.append(summary)

Token indices sequence length is longer than the specified maximum sequence length for this model (595 > 512). Running this sequence through the model will result in indexing errors


In [68]:
for i, s in enumerate(chunk_summaries, 1):
    print(f"Chunk {i} Summary:\n{s}\n")

Chunk 1 Summary:
the e-ink screen almost feels like a page of a paperback when you swipe . the battery is rated to last 6-8 weeks on average between the various tech sites, which trumps my ipad mini .

Chunk 2 Summary:
the paperwhite is a great reader indoor or out no light to full light . it's light weight, easy to read inside or outside . the screen's front-lit display allows me to read under all lighting conditions .

Chunk 3 Summary:
the Kindle Paperwhite is reliable, holds a charge for weeks, is lightweight and easy to hold . it has a few less LEDs for backlighting, no flush glass cover, and a tiny bit heavier/bigger than the Voyage .

Chunk 4 Summary:
the Kindle Voyage is the ONLY Kindle to feature the auto-brightness feature . it has nice features, such as dictionary, where you can quickly check unfamiliar words while reading, create notes, or write and read reviews .



### Combine all chunk summaries

In [69]:
combined_summary = "summarize: " + " ".join(chunk_summaries)

### Generate the final summary

In [70]:
final_summary = summarizer(
    combined_summary,
    max_length=180,
    min_length=80,
    do_sample=False
)[0]["summary_text"]

print(final_summary)

the battery is rated to last 6-8 weeks on average between the various tech sites, which trumps my ipad mini . the paperwhite is a great reader indoor or out no light to full light . it's light weight, easy to read inside or outside . this screen's front-lit display allows me to read under all lighting conditions .


### Save the Result

In [71]:
results = pd.DataFrame({
    "Meta_Category": [category],
    "t5_Summary": [final_summary]
})

results

,Meta_Category,t5_Summary
0,Tablets & eReaders,the battery is rated to last 6-8 weeks on aver...


### Summarize all 4 categories

In [72]:
t5_results = []

for category in selected_reviews["Meta_Category"].unique():

    texts = selected_reviews.loc[
        selected_reviews["Meta_Category"] == category,
        "reviews.text"
    ]

    selected_sentences = top_representative_sentences(texts, n=40)

    chunks = chunk_sentences(selected_sentences, chunk_size=10)

    chunk_summaries = []

    for chunk in chunks:
        text = " ".join(chunk)

        summary = summarizer(
            text,
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]

        chunk_summaries.append(summary)

    combined_summary = " ".join(chunk_summaries)

    final_summary = summarizer(
        combined_summary,
        max_length=180,
        min_length=80,
        do_sample=False
    )[0]["summary_text"]

    t5_results.append({
        "Meta_Category": category,
        "t5_Summary": final_summary
    })

t5_results = pd.DataFrame(t5_results)

t5_results

,Meta_Category,t5_Summary
0,Tablets & eReaders,"this new Kindle is awesome for reading only, a..."
1,Smart Home & Alexa Devices,my favorite part is being able to request any ...
2,Electronics & Streaming Devices,amazon fire TV's work fantastically and we dec...
3,Batteries & Household Essentials,amazonBasics products are great for middle inc...


### Save it as csv

In [73]:
t5_results.to_csv("t5_summaries.csv", index=False)

In [74]:
summarizer.model.save_pretrained("t5_model")
summarizer.tokenizer.save_pretrained("t5_model")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'min_length': 30, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3}


('t5_model/tokenizer_config.json',
 't5_model/special_tokens_map.json',
 't5_model/spiece.model',
 't5_model/added_tokens.json',
 't5_model/tokenizer.json')

In [75]:
t5_results.to_pickle("t5_summaries.pkl")